In [5]:
import sys
print(sys.executable)

import pyspark
print(pyspark.__version__)

/Users/sona/Olist_pandas_assigment/.venv/bin/python
4.2.0


In [1]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@25/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [7]:
import os
import subprocess

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

JAVA_HOME: /opt/homebrew/opt/openjdk@25/libexec/openjdk.jdk/Contents/Home
openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment Homebrew (build 25.0.4.1)
OpenJDK 64-Bit Server VM Homebrew (build 25.0.4.1, mixed mode, sharing)



In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataKern-Olist-Spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/sona/Olist_pandas_assigment/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/08 10:02:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("App name:", spark.sparkContext.appName)

Spark version: 4.2.0
Master: local[*]
App name: DataKern-Olist-Spark


In [9]:
import os

print("Working directory:")
print(os.getcwd())

print("\nFile exists:")
print(os.path.exists("data/raw/olist_orders_dataset.csv"))

Working directory:
/Users/sona/Olist_pandas_assigment/notebooks

File exists:
False


In [10]:
orders_inferred = spark.read.csv(
    "../data/raw/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

In [11]:
orders_inferred.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [13]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

In [14]:
StructField(
    "order_delivered_customer_date",
    TimestampType(),
    True
)

StructField('order_delivered_customer_date', TimestampType(), True)

In [15]:
orders_schema

StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

In [16]:
orders = spark.read.csv(
    "../data/raw/olist_orders_dataset.csv",
    header=True,
    schema=orders_schema
)

In [17]:
orders_schema

StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

In [18]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [19]:
orders_inferred.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [20]:
customers = spark.read.csv(
    "../data/raw/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

In [21]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [22]:
order_items = spark.read.csv(
    "../data/raw/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True

)

In [23]:
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [24]:
products = spark.read.csv(
    "../data/raw/olist_products_dataset.csv",
    header=True,
    inferSchema=True

)

In [25]:
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [28]:
category_translation = spark.read.csv(
    "../data/raw/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

In [29]:
category_translation.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [30]:
orders.count()


99441

## First action — orders.count()
The orders DataFrame contains 99,441 rows. Unlike printSchema(), count() is an action, so Spark had to execute the plan and read the input data. I then inspected the Spark UI to observe the job, stage, and task information.


In [32]:
orders.count()

99441

In [34]:
orders

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]

In [35]:
orders_selected = orders.select(
    "order_id",
    "customer_id",
    "order_status"
)

In [36]:
orders_selected.explain()

== Physical Plan ==
FileScan csv [order_id#25,customer_id#26,order_status#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [37]:
delivered_orders = orders.filter(
    orders.order_status == "delivered"
)

In [38]:
delivered_orders.explain()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#27) AND (order_status#27 = delivered))
+- FileScan csv [order_id#25,customer_id#26,order_status#27,order_purchase_timestamp#28,order_approved_at#29,order_delivered_carrier_date#30,order_delivered_customer_date#31,order_estimated_delivery_date#32] Batched: false, DataFilters: [isnotnull(order_status#27), (order_status#27 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,...




In [39]:
delivered_orders.count()

96478

## delivered_orders.count()
- Job: 14
- Status: SUCCEEDED
- Duration: 0.1 s
- Completed stages: 1
- Stage: 16
- Tasks: 5/5 succeeded
- Input: 17.1 MiB
- Shuffle write: 295 B
- Shuffle read: blank/none shown
The important Spark concept here is partitions → tasks.
Spark processed the input using 5 tasks, which means this stage had 5 input partitions to process. Each task works on one partition.
Also notice that the job has only 1 stage. That makes sense for this operation because your delivered_orders filtering is a narrow transformation; it does not require redistributing the data across the cluster.
The tiny 295 B shuffle write is worth noting, but don't interpret that as the filter itself performing a large shuffle. The main computation here is still a simple scan/filter/count.
Result: 96,478 delivered orders.
Execution: 1 stage with 5 tasks.
Interpretation: the 5 tasks reflect the input partitioning, and the filter is a narrow transformation

In [40]:
orders_selected = orders.select(
    "order_id",
    "customer_id",
    "order_status"
)

In [41]:
orders_selected.explain()

== Physical Plan ==
FileScan csv [order_id#25,customer_id#26,order_status#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [42]:
orders_selected = orders.select(
    "order_id",
    "customer_id",
    "order_status"
)

In [43]:
orders_selected.explain()

== Physical Plan ==
FileScan csv [order_id#25,customer_id#26,order_status#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [45]:
orders_selected.explain()

== Physical Plan ==
FileScan csv [order_id#25,customer_id#26,order_status#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [46]:
orders_selected.count()

99441

### orders_selected.count()
orders_selected.count() returned 99,441
the select() transformation itself did not create an execution job
the .count() action triggered execution
Job 16 had 1 stage and 5 successful tasks
the physical plan showed a FileScan containing only the three selected columns
select(): The select() operation was lazy and did not execute immediately. When count() was called, Spark executed the plan. The result was 99,441 rows. The job used 1 stage and 5 tasks, showing that the work was performed across the input partitions. The physical plan showed that Spark scanned the selected columns.


In [47]:
delivered_orders = orders.filter(
    orders.order_status == "delivered"
)

In [48]:
delivered_orders.explain()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#27) AND (order_status#27 = delivered))
+- FileScan csv [order_id#25,customer_id#26,order_status#27,order_purchase_timestamp#28,order_approved_at#29,order_delivered_carrier_date#30,order_delivered_customer_date#31,order_estimated_delivery_date#32] Batched: false, DataFilters: [isnotnull(order_status#27), (order_status#27 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,...




In [49]:
delivered_orders.count()

96478

### filter()
The important Spark concept here is partition → task:
- Spark processed the input using 5 tasks.
- The input was divided across those 5 partitions.
- The filter() itself does not require a large shuffle, because each partition can independently check whether order_status == "delivered".
- The tiny 295 B shuffle write comes from the count() aggregation, not from the filter redistributing the full dataset.

### withColumn()

In [50]:
from pyspark.sql.functions import year

delivered_with_year = delivered_orders.withColumn(
    "purchase_year",
    year("order_purchase_timestamp")
)

In [51]:
delivered_with_year.explain()

== Physical Plan ==
*(1) Project [order_id#25, customer_id#26, order_status#27, order_purchase_timestamp#28, order_approved_at#29, order_delivered_carrier_date#30, order_delivered_customer_date#31, order_estimated_delivery_date#32, year(cast(order_purchase_timestamp#28 as date)) AS purchase_year#182]
+- *(1) Filter (isnotnull(order_status#27) AND (order_status#27 = delivered))
   +- FileScan csv [order_id#25,customer_id#26,order_status#27,order_purchase_timestamp#28,order_approved_at#29,order_delivered_carrier_date#30,order_delivered_customer_date#31,order_estimated_delivery_date#32] Batched: false, DataFilters: [isnotnull(order_status#27), (order_status#27 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestam

In [52]:
delivered_with_year.select("purchase_year").show(5)

+-------------+
|purchase_year|
+-------------+
|         2017|
|         2018|
|         2018|
|         2017|
|         2018|
+-------------+
only showing top 5 rows


In [4]:
spark.version

'4.2.0'

In [9]:
print("orders_schema exists:", "orders_schema" in globals())

orders_schema exists: False


In [10]:
"orders_schema" in globals()

False

In [11]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

In [12]:
print("orders_schema" in globals())

True


In [14]:
orders = spark.read.csv(
    "../data/raw/olist_orders_dataset.csv",
    header=True,
    schema=orders_schema
)

In [15]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [16]:
customers = spark.read.csv(
    "../data/raw/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [17]:
order_items = spark.read.csv(
    "../data/raw/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [18]:
products = spark.read.csv(
    "../data/raw/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [20]:
category_translation = spark.read.csv(
    "../data/raw/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

category_translation.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [21]:
orders.count()

99441

In [22]:
orders_slim = orders.select(
    "order_id",
    "customer_id",
    "order_status"
)

orders_slim.explain()

== Physical Plan ==
FileScan csv [order_id#0,customer_id#1,order_status#2] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [23]:
delivered_orders = orders_slim.filter(
    orders_slim.order_status == "delivered"
)

delivered_orders.explain()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#2) AND (order_status#2 = delivered))
+- FileScan csv [order_id#0,customer_id#1,order_status#2] Batched: false, DataFilters: [isnotnull(order_status#2), (order_status#2 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string>




In [25]:
orders_slim = orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)

delivered_orders = orders_slim.filter(
    orders_slim.order_status == "delivered"
)

In [26]:
delivered_with_year = delivered_orders.withColumn(
    "purchase_year",
    year("order_purchase_timestamp")
)

delivered_with_year.explain()

== Physical Plan ==
*(1) Project [order_id#0, customer_id#1, order_status#2, order_purchase_timestamp#3, year(cast(order_purchase_timestamp#3 as date)) AS purchase_year#113]
+- *(1) Filter (isnotnull(order_status#2) AND (order_status#2 = delivered))
   +- FileScan csv [order_id#0,customer_id#1,order_status#2,order_purchase_timestamp#3] Batched: false, DataFilters: [isnotnull(order_status#2), (order_status#2 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/sona/Olist_pandas_assigment/data/raw/olist_orders_dataset...., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp>




In [27]:
orders_with_items = delivered_with_year.join(
    order_items,
    on="order_id",
    how="inner"
)

orders_with_items.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#0, customer_id#1, order_status#2, order_purchase_timestamp#3, purchase_year#113, order_item_id#48, product_id#49, seller_id#50, shipping_limit_date#51, price#52, freight_value#53]
   +- SortMergeJoin [order_id#0], [order_id#47], Inner
      :- Sort [order_id#0 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(order_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=158]
      :     +- Project [order_id#0, customer_id#1, order_status#2, order_purchase_timestamp#3, year(cast(order_purchase_timestamp#3 as date)) AS purchase_year#113]
      :        +- Filter ((isnotnull(order_status#2) AND (order_status#2 = delivered)) AND isnotnull(order_id#0))
      :           +- FileScan csv [order_id#0,customer_id#1,order_status#2,order_purchase_timestamp#3] Batched: false, DataFilters: [isnotnull(order_status#2), (order_status#2 = delivered), isnotnull(order_id#0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[fi

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataKern-Olist-Spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/sona/Olist_pandas_assigment/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/21 10:09:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


In [3]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

orders = spark.read.csv(
    "../data/raw/olist_orders_dataset.csv",
    header=True,
    schema=orders_schema
)

print("Orders loaded:", orders.count())

Orders loaded: 99441


In [4]:
order_items = spark.read.csv(
    "../data/raw/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

print("Order items:", order_items.count())

Order items: 112650


In [5]:
customers = spark.read.csv(
    "../data/raw/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

print("Customers:", customers.count())

Customers: 99441


In [6]:
products = spark.read.csv(
    "../data/raw/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

print("Products:", products.count())

Products: 32951


In [7]:
category_translation = spark.read.csv(
    "../data/raw/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

print("Category translations:", category_translation.count())

Category translations: 71


In [8]:
from pyspark.sql.functions import year

orders_slim = orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)

delivered_orders = orders_slim.filter(
    orders_slim.order_status == "delivered"
)

delivered_with_year = delivered_orders.withColumn(
    "purchase_year",
    year("order_purchase_timestamp")
)

orders_with_items = delivered_with_year.join(
    order_items,
    on="order_id",
    how="inner"
)

print("Joined rows:", orders_with_items.count())

Joined rows: 110197


In [9]:
from pyspark.sql.functions import month

orders_with_month = orders_with_items.withColumn(
    "purchase_month",
    month("order_purchase_timestamp")
)

orders_with_month.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#0, customer_id#1, order_status#2, order_purchase_timestamp#3, purchase_year#151, order_item_id#38, product_id#39, seller_id#40, shipping_limit_date#41, price#42, freight_value#43, month(cast(order_purchase_timestamp#3 as date)) AS purchase_month#171]
   +- SortMergeJoin [order_id#0], [order_id#37], Inner
      :- Sort [order_id#0 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(order_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=384]
      :     +- Project [order_id#0, customer_id#1, order_status#2, order_purchase_timestamp#3, year(cast(order_purchase_timestamp#3 as date)) AS purchase_year#151]
      :        +- Filter ((isnotnull(order_status#2) AND (order_status#2 = delivered)) AND isnotnull(order_id#0))
      :           +- FileScan csv [order_id#0,customer_id#1,order_status#2,order_purchase_timestamp#3] Batched: false, DataFilters: [isnotnull(order_status#2), (order_status#2 = delivered), isnot

In [10]:
from pyspark.sql.functions import (
    countDistinct,
    sum as spark_sum,
    round as spark_round
)

monthly_summary = (
    orders_with_month
    .groupBy("purchase_year", "purchase_month")
    .agg(
        countDistinct("order_id").alias("distinct_orders"),
        spark_round(spark_sum("price"), 2).alias("total_item_sales")
    )
    .orderBy("purchase_year", "purchase_month")
)

monthly_summary.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [purchase_year#151 ASC NULLS FIRST, purchase_month#171 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(purchase_year#151 ASC NULLS FIRST, purchase_month#171 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=450]
      +- HashAggregate(keys=[purchase_year#151, purchase_month#171], functions=[sum(price#42), count(distinct order_id#0)])
         +- Exchange hashpartitioning(purchase_year#151, purchase_month#171, 200), ENSURE_REQUIREMENTS, [plan_id=447]
            +- HashAggregate(keys=[purchase_year#151, purchase_month#171], functions=[merge_sum(price#42), partial_count(distinct order_id#0)])
               +- HashAggregate(keys=[purchase_year#151, purchase_month#171, order_id#0], functions=[merge_sum(price#42)])
                  +- Exchange hashpartitioning(purchase_year#151, purchase_month#171, order_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=443]
                     +- HashAggregate(keys=[purchase_year#151,

In [11]:
monthly_summary.show(24, truncate=False)

+-------------+--------------+---------------+----------------+
|purchase_year|purchase_month|distinct_orders|total_item_sales|
+-------------+--------------+---------------+----------------+
|2016         |9             |1              |134.97          |
|2016         |10            |265            |40325.11        |
|2016         |12            |1              |10.9            |
|2017         |1             |750            |111798.36       |
|2017         |2             |1653           |234223.4        |
|2017         |3             |2546           |359198.85       |
|2017         |4             |2303           |340669.68       |
|2017         |5             |3546           |489338.25       |
|2017         |6             |3135           |421923.37       |
|2017         |7             |3872           |481604.52       |
|2017         |8             |4193           |554699.7        |
|2017         |9             |4150           |607399.67       |
|2017         |10            |4478      

In [12]:
total_rows = order_items.count()

distinct_rows = (
    order_items
    .dropDuplicates(["order_id", "order_item_id"])
    .count()
)

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate keys:", total_rows - distinct_rows)

Total rows: 112650
Distinct rows: 112650
Duplicate keys: 0


In [13]:
order_items_clean = order_items.dropDuplicates(
    ["order_id", "order_item_id"]
)

print("Cleaned rows:", order_items_clean.count())

Cleaned rows: 112650


In [14]:
orders_with_items.cache()

orders_with_items.count()

110197

In [15]:
print("Original partitions:", orders_with_items.rdd.getNumPartitions())

orders_repartitioned = orders_with_items.repartition(8)

print("After repartition:", orders_repartitioned.rdd.getNumPartitions())

Original partitions: 200
After repartition: 8


In [16]:
orders_coalesced = orders_repartitioned.coalesce(2)

print(
    "After coalesce:",
    orders_coalesced.rdd.getNumPartitions()
)

After coalesce: 2


In [17]:
monthly_summary.write.mode("overwrite").parquet(
    "../output/monthly_sales_summary"
)

In [18]:
saved_summary = spark.read.parquet(
    "../output/monthly_sales_summary"
)

saved_summary.show(5)
saved_summary.printSchema()

+-------------+--------------+---------------+----------------+
|purchase_year|purchase_month|distinct_orders|total_item_sales|
+-------------+--------------+---------------+----------------+
|         2016|             9|              1|          134.97|
|         2016|            10|            265|        40325.11|
|         2016|            12|              1|            10.9|
|         2017|             1|            750|       111798.36|
|         2017|             2|           1653|        234223.4|
+-------------+--------------+---------------+----------------+
only showing top 5 rows
root
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- distinct_orders: long (nullable = true)
 |-- total_item_sales: double (nullable = true)



In [19]:
items_with_products = order_items_clean.join(
    products,
    on="product_id",
    how="left"
)

items_with_products.select(
    "order_id",
    "product_id",
    "product_category_name",
    "price"
).show(5, truncate=False)

+--------------------------------+--------------------------------+---------------------+-----+
|order_id                        |product_id                      |product_category_name|price|
+--------------------------------+--------------------------------+---------------------+-----+
|00018f77f2f0320c557190d7a144bdd3|e5f2d52b802189ee658865ca93d83a8f|pet_shop             |239.9|
|00061f2a7bc09da83e415a52dc8a4af1|d63c1011f49d98b976c352955b1c4bea|beleza_saude         |59.99|
|0014ae671de39511f7575066200733b7|23365beed316535b4105bd800c46670e|telefonia            |16.5 |
|0015ebb40fb17286bea51d4607c4733c|50fd2b788dc166edd20512370dac54df|eletronicos          |21.9 |
|001ab0a7578dd66cd4b0a71f5b6e1e41|0b0172eb0fd18479d29c3bc122c058c2|eletronicos          |24.89|
+--------------------------------+--------------------------------+---------------------+-----+
only showing top 5 rows


In [20]:
items_with_category = items_with_products.join(
    category_translation,
    on="product_category_name",
    how="left"
)

items_with_category.select(
    "product_category_name",
    "product_category_name_english",
    "price"
).show(10, truncate=False)

+---------------------------------+-----------------------------+-----+
|product_category_name            |product_category_name_english|price|
+---------------------------------+-----------------------------+-----+
|pet_shop                         |pet_shop                     |239.9|
|beleza_saude                     |health_beauty                |59.99|
|telefonia                        |telephony                    |16.5 |
|eletronicos                      |electronics                  |21.9 |
|eletronicos                      |electronics                  |24.89|
|cama_mesa_banho                  |bed_bath_table               |69.9 |
|construcao_ferramentas_iluminacao|construction_tools_lights    |29.99|
|utilidades_domesticas            |housewares                   |78.0 |
|perfumaria                       |perfumery                    |155.0|
|cama_mesa_banho                  |bed_bath_table               |53.98|
+---------------------------------+-----------------------------

In [21]:
finance_data = items_with_category.join(
    orders.select(
        "order_id",
        "order_status",
        "order_purchase_timestamp"
    ),
    on="order_id",
    how="inner"
)

finance_data.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "product_category_name_english",
    "price"
).show(5, truncate=False)

+--------------------------------+------------+------------------------+-----------------------------+-----+
|order_id                        |order_status|order_purchase_timestamp|product_category_name_english|price|
+--------------------------------+------------+------------------------+-----------------------------+-----+
|00018f77f2f0320c557190d7a144bdd3|delivered   |2017-04-26 10:53:06     |pet_shop                     |239.9|
|00061f2a7bc09da83e415a52dc8a4af1|delivered   |2018-03-24 22:16:10     |health_beauty                |59.99|
|0014ae671de39511f7575066200733b7|delivered   |2017-05-22 13:49:03     |telephony                    |16.5 |
|0015ebb40fb17286bea51d4607c4733c|delivered   |2018-01-14 09:01:36     |electronics                  |21.9 |
|001ab0a7578dd66cd4b0a71f5b6e1e41|delivered   |2017-12-27 00:38:47     |electronics                  |24.89|
+--------------------------------+------------+------------------------+-----------------------------+-----+
only showing top 5 

In [22]:
from pyspark.sql.functions import col, date_trunc

finance_delivered = finance_data.filter(
    col("order_status") == "delivered"
)

finance_with_month = finance_delivered.withColumn(
    "reporting_month",
    date_trunc("month", col("order_purchase_timestamp"))
)

finance_with_month.select(
    "reporting_month",
    "product_category_name_english",
    "price"
).show(5, truncate=False)

+-------------------+-----------------------------+-----+
|reporting_month    |product_category_name_english|price|
+-------------------+-----------------------------+-----+
|2017-04-01 00:00:00|pet_shop                     |239.9|
|2018-03-01 00:00:00|health_beauty                |59.99|
|2017-05-01 00:00:00|telephony                    |16.5 |
|2018-01-01 00:00:00|electronics                  |21.9 |
|2017-12-01 00:00:00|electronics                  |24.89|
+-------------------+-----------------------------+-----+
only showing top 5 rows


In [23]:
from pyspark.sql.functions import (
    countDistinct,
    sum as spark_sum,
    round as spark_round
)

finance_summary = (
    finance_with_month
    .groupBy(
        "reporting_month",
        "product_category_name_english"
    )
    .agg(
        spark_round(
            spark_sum("price"), 2
        ).alias("total_item_sales"),
        countDistinct("order_id").alias("distinct_orders")
    )
    .withColumnRenamed(
        "product_category_name_english",
        "product_category"
    )
    .orderBy("reporting_month", "product_category")
)

finance_summary.show(20, truncate=False)

+-------------------+------------------------+----------------+---------------+
|reporting_month    |product_category        |total_item_sales|distinct_orders|
+-------------------+------------------------+----------------+---------------+
|2016-09-01 00:00:00|health_beauty           |134.97          |1              |
|2016-10-01 00:00:00|NULL                    |65.89           |2              |
|2016-10-01 00:00:00|air_conditioning        |1349.29         |4              |
|2016-10-01 00:00:00|audio                   |156.99          |2              |
|2016-10-01 00:00:00|auto                    |1128.26         |8              |
|2016-10-01 00:00:00|baby                    |1371.17         |9              |
|2016-10-01 00:00:00|bed_bath_table          |478.99          |5              |
|2016-10-01 00:00:00|computers_accessories   |669.02          |12             |
|2016-10-01 00:00:00|consoles_games          |3619.36         |7              |
|2016-10-01 00:00:00|cool_stuff         

In [24]:
print("Finance summary rows:", finance_summary.count())

finance_summary.filter(
    finance_summary.product_category.isNull()
).show(10, truncate=False)

Finance summary rows: 1264
+-------------------+----------------+----------------+---------------+
|reporting_month    |product_category|total_item_sales|distinct_orders|
+-------------------+----------------+----------------+---------------+
|2016-10-01 00:00:00|NULL            |65.89           |2              |
|2017-01-01 00:00:00|NULL            |1603.64         |10             |
|2017-02-01 00:00:00|NULL            |8521.11         |50             |
|2017-03-01 00:00:00|NULL            |6457.17         |54             |
|2017-04-01 00:00:00|NULL            |8708.17         |63             |
|2017-05-01 00:00:00|NULL            |10948.33        |83             |
|2017-06-01 00:00:00|NULL            |5032.39         |51             |
|2017-07-01 00:00:00|NULL            |2967.86         |52             |
|2017-08-01 00:00:00|NULL            |4968.45         |64             |
|2017-09-01 00:00:00|NULL            |5774.15         |61             |
+-------------------+----------------

In [25]:
operations_data = orders.join(
    customers.select(
        "customer_id",
        "customer_state"
    ),
    on="customer_id",
    how="left"
)

operations_data.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "customer_state"
).show(5, truncate=False)

+--------------------------------+------------+------------------------+-----------------------------+-----------------------------+--------------+
|order_id                        |order_status|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|customer_state|
+--------------------------------+------------+------------------------+-----------------------------+-----------------------------+--------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |2017-10-02 10:56:33     |2017-10-10 21:25:13          |2017-10-18 00:00:00          |SP            |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |2018-07-24 20:41:37     |2018-08-07 15:27:45          |2018-08-13 00:00:00          |BA            |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |2018-08-08 08:38:49     |2018-08-17 18:06:29          |2018-09-04 00:00:00          |GO            |
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |2017-11-18 19:28:06     |2017-12-02 00:28:42          |2017-12-15

In [26]:
from pyspark.sql.functions import col, when, date_trunc

operations_classified = operations_data.withColumn(
    "delivery_status",
    when(
        col("order_delivered_customer_date").isNull(),
        "not_delivered"
    ).when(
        col("order_delivered_customer_date") >
        col("order_estimated_delivery_date"),
        "late"
    ).otherwise("on_time")
)

In [27]:
operations_classified.groupBy(
    "delivery_status"
).count().show()

+---------------+-----+
|delivery_status|count|
+---------------+-----+
|           late| 7827|
|        on_time|88649|
|  not_delivered| 2965|
+---------------+-----+



In [28]:
operations_with_month = operations_classified.withColumn(
    "reporting_month",
    date_trunc("month", col("order_purchase_timestamp"))
)

In [30]:
from pyspark.sql.functions import (
    col, when, sum as spark_sum, round as spark_round
)

operations_summary = (
    operations_with_month
    .groupBy("reporting_month", "customer_state")
    .agg(
        spark_sum(
            when(col("delivery_status").isin("late", "on_time"), 1)
            .otherwise(0)
        ).alias("delivered_orders"),
        spark_sum(
            when(col("delivery_status") == "late", 1)
            .otherwise(0)
        ).alias("late_orders")
    )
    .withColumn(
        "late_rate",
        when(
            col("delivered_orders") > 0,
            spark_round(
                col("late_orders") / col("delivered_orders"), 4
            )
        ).otherwise(None)
    )
    .orderBy("reporting_month", "customer_state")
)

operations_summary.show(20, truncate=False)

+-------------------+--------------+----------------+-----------+---------+
|reporting_month    |customer_state|delivered_orders|late_orders|late_rate|
+-------------------+--------------+----------------+-----------+---------+
|2016-09-01 00:00:00|RR            |0               |0          |NULL     |
|2016-09-01 00:00:00|RS            |0               |0          |NULL     |
|2016-09-01 00:00:00|SP            |1               |1          |1.0      |
|2016-10-01 00:00:00|AL            |1               |0          |0.0      |
|2016-10-01 00:00:00|BA            |3               |0          |0.0      |
|2016-10-01 00:00:00|CE            |6               |0          |0.0      |
|2016-10-01 00:00:00|DF            |6               |0          |0.0      |
|2016-10-01 00:00:00|ES            |3               |0          |0.0      |
|2016-10-01 00:00:00|GO            |7               |0          |0.0      |
|2016-10-01 00:00:00|MA            |4               |1          |0.25     |
|2016-10-01 

In [31]:
finance_summary.write.mode("overwrite").parquet(
    "../output/finance_summary"
)

In [32]:
operations_summary.write.mode("overwrite").parquet(
    "../output/operations_summary"
)

In [33]:
finance_check = spark.read.parquet("../output/finance_summary")
operations_check = spark.read.parquet("../output/operations_summary")

print("Finance rows:", finance_check.count())
print("Operations rows:", operations_check.count())

finance_check.printSchema()
operations_check.printSchema()

Finance rows: 1264
Operations rows: 565
root
 |-- reporting_month: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- total_item_sales: double (nullable = true)
 |-- distinct_orders: long (nullable = true)

root
 |-- reporting_month: timestamp (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- late_orders: long (nullable = true)
 |-- late_rate: double (nullable = true)



## Spark UI Observations


| Operation | Observation | Explanation |
|---|---|---|
| Orders count | 99,441 orders; 5 tasks observed in an earlier execution | Spark processed the input partitions to calculate the count. |
| Delivered orders | 96,478 orders | Filtering kept delivered orders; the filter itself does not normally require a shuffle. |
| Repartition | 200 partitions changed to 8 | Repartition redistributes data and normally involves a shuffle. |
| Coalesce | 8 partitions changed to 2 | Coalesce reduced the number of partitions. |
| Finance cache | 200 cached partitions; 17.7 MiB in memory | The cached DataFrame can be reused without recomputing all its transformations. |
| Finance output | 1,264 rows | Monthly category-level sales summary. |
| Operations output | 565 rows | Monthly state-level delivery summary. |
| Parquet write | Both output writes completed successfully | The summaries were saved in Parquet format. |